# Serve Qwen through a NIM container

**You are here → 03 — Qwen through NIM.**

**Main workflow:** understand a local AI request → optimize Qwen → optimize Nemotron.
**Optional extensions:** model-specific NIM deployment; separate CPU portability branch.

| **Step** | **Notebook** | **Purpose** |
|---|---|---|
| 0. Understand the request | [00 — How a local AI request works](00_local_serving.ipynb) | Follow a question through schema context, model serving and generated SQL, then evaluate the results to establish a quality baseline. |
| 1. Qwen optimization | [01 — Qwen: caching and quantization](01_qwen_cache_quantization.ipynb) | Compare FP16/AWQ W4A16 and prefix caching, with cold and warm requests reported separately. |
| 2. Nemotron optimization | [02 — Nemotron optimization](02_nemotron_optimization.ipynb) | Test schema context, supported prefix caching, concurrent requests, scheduler capacity and output budgets. |
| **→ 3. Qwen NIM (optional) (current)** | [03 — Qwen through NIM](03_qwen_nim.ipynb) | Deploy Qwen in a container, check the plan format, and compare schema context and concurrent requests. |
| 4. Nemotron NIM (optional) | [04 — Nemotron through NIM](04_nemotron_nim.ipynb) | Deploy the hybrid model through NIM and compare schema context and request concurrency. |
| Optional CPU portability | [05 — Qwen with ExecuTorch](05_edge_portability_executorch.ipynb) | Follow Qwen3-4B conversion and CPU export, then inspect a generated SQL answer and its timing. |
| Optional compatibility study | [06 — Nemotron with ExecuTorch](06_nemotron_executorch.ipynb) | Check whether the installed exporter accepts Nemotron and identify what a working port still needs. |

Chapter 01 served Qwen3-4B through a native vLLM installation. Here we'll serve
the model through **NVIDIA NIM**, which packages a serving backend and its
dependencies in a Docker container. A **container** gives us a packaged runtime
to start, configure and stop; a **profile** selects a deployment setup for the
model and GPU.

We'll use a vLLM-backed NIM container, check that the application accepts the
model's response format, then compare full and retrieved schema context and
one, two and four concurrent requests. The first failed SQL run will help us
see why a working server is only one part of a working application.

You can work through this chapter in two ways. **Read the completed experiment:**
choose **Run All** in the [notebook environment](../SETUP.md) to display saved
measurements. **Run a new experiment:** use the [Live setup](../tutorial/RUN_ALL.md)
with a compatible GPU. The default reads results only; it downloads no weights
and starts no inference server.

The GPU measurements in this chapter used an **NVIDIA Quadro RTX 5000 with
16 GiB VRAM**. We use synthetic clinical-trial questions to connect speed with
SQL outcomes. These experiments help us understand this workload, rather than
predict performance for every application.


## 1. Choose how to run the experiment

The next cell loads the configuration. Check the printed mode so you know
whether later cells are reading a completed run or collecting new results.

| Mode | What the cells do |
|---|---|
| **Replay (default)** | Read the supplied results and rebuild their tables; no inference runs |
| Plan | Show settings and commands for a new experiment; no inference runs |
| Live | Start the configured server, make requests and save new measurements |

For Plan or Live, create `tutorial/config/nim-qwen.local.json` using
[the run guide](../tutorial/RUN_ALL.md). Put your paths and settings there. Editing
that config affects a new run; it doesn't change the measurements read in Replay.

We use `Qwen/Qwen3-4B` through a vLLM-backed, model-free NIM container, with weights loaded as **FP16**.
This NIM recipe uses unquantized weights; it does not run the AWQ matrix from chapter 01.

There are **four distinct questions**, repeated five times for serving measurements.
The separate application evaluation includes planning, SQL checks, execution, and up
to one repair. Twenty repeated requests are not twenty independent quality examples.


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/clinical_trial_mvp').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from IPython.display import JSON, Markdown, display
from clinical_trial_mvp.tutorial.workflow import Study
from clinical_trial_mvp.tutorial.teaching import server_example, request_example, response_example, interpret
if 'study' in globals():
    study.close()
study = Study('nim', 'qwen', root=ROOT)
results = study.results
print('Mode:', study.mode)
print('Model:', 'Qwen/Qwen3-4B')
print(study.prepare())


Mode: replay
Model: Qwen/Qwen3-4B
Replay: recorded Dell measurements; no model download, server, GPU or network calls.


## 2. What changes when we package the server?

Our NIM container runs **vLLM**. Packaging the server doesn't automatically
convert the model to TensorRT or change its weights. We'll keep one deployment
fixed, then change schema context and request concurrency to see how the
application behaves through this endpoint.


For a new Live run, list the profiles compatible with your model, container
image and GPU, then choose a runnable base-model profile without LoRA adapters.
**TP1/PP1** means one tensor-parallel shard and one pipeline stage; this experiment
uses one GPU. Start the container and check a complete application request before
measuring speed. Follow the [NIM setup and profile guide](../tutorial/NIM_MVP4.md).

Our first Qwen run returned all four responses, but every plan used an object
for `filters` where the application expected a string. No application SQL executed.
Enabling the JSON-schema constraint led to 3/4 executions in the rerun. Here,
**JSON schema** describes the required fields and types in the generated plan;
it is different from the **database schema** we provide as context. The reusable
Qwen configuration enables that output-format constraint.

Don't compare NIM and native timings as if packaging were the only difference.
Thinking behavior is not yet verified to match; the same `/no_think` text does not
prove parity. Check model, prompts, data/date, contracts, backend and generation
settings before drawing a causal speed conclusion. AWQ and controlled-cache
extensions are not part of this chapter's implemented core recipe.


## 3. Check that the model can complete a SQL request

Before comparing speed, check that the container and its request format can work
with the application. This **smoke test** sends a few questions through planning,
SQL generation and execution. It can catch an integration failure before we
spend time on a larger run; it doesn't replace the later quality comparison.

In Replay, the next cell shows the saved smoke result when one is available.
Plan makes no model calls. Live runs the check and stops dependent experiments
if the required check fails. Read the status and any error before continuing.


In [2]:
smoke = study.smoke()
if isinstance(smoke, dict):
    print(smoke.get('_recorded', {}).get('label', study.mode), smoke.get('status'), smoke.get('reason', ''))
else:
    print(smoke)


recorded Dell measurement completed 


## 4. Run the comparisons

We'll establish a baseline, then change a selected setting in each comparison.
Before running a cell, note what changes and what stays fixed. Afterward, compare
request timing with the generated SQL and its result check. An experiment finishing
successfully doesn't mean every answer it produced was correct.


### Establish the full-schema baseline

Let's start with one client and the full schema. Read the smoke result before
using this row as a reference. A ready HTTP endpoint isn't enough: the generated
plan must also satisfy the application contract.


In [3]:
result = study.run('baseline')
print(result.get('_recorded', {}).get('label', study.mode), 'baseline:', result['status'])
display(Markdown(interpret(study, 'baseline')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'baseline')))


recorded Dell measurement baseline: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/nim/runs/20261107T215646_526562Z.json](../experiment-tracker/mvp4/nim/runs/20261107T215646_526562Z.json)

**Prediction:** Establish executable SQL and separate application quality from request speed.

**Observation:** Application result match: 2/4; SQL executed: 3/4. Request E2E p50/p95: 6.200/7.775 s; aggregate output: 24.136 tok/s.

**Explanation:** Transport timing and application correctness measure different stages.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Use this baseline to judge useful completed tasks, not latency alone.

### Connect the server settings to the request

Start with the server: which checkpoint is loaded, what precision does it use,
and what context and scheduling limits apply? The next cell displays those settings
and a launch command so you can connect the experiment config to the serving process.
In Plan, it shows the proposed settings for your new run.

The [NIM deployment helper](../src/clinical_trial_mvp/tutorial/nim_study.py) contains the implementation.


In [4]:
display(JSON(server_example(study, 'baseline')))


<IPython.core.display.JSON object>

**Next, inspect the request.** The model receives a prompt, generation limits
and, where enabled, a JSON response schema. Look for the shared database context,
the question-specific text and the requested output format in the next cell.
This lets us see what the application asks the model to do before judging its answer.

This inspection displays the example without sending another request.
The [request builder](../src/clinical_trial_mvp/serving/client.py) defines the payload.


In [5]:
display(JSON(request_example(study, 'baseline')))


<IPython.core.display.JSON object>

**Finally, follow the answer.** The next cell shows the generated plan and SQL,
any repair attempt, and the comparison with the reference query. Check the
question's filters, grouping and counted population as well as the score.
Matching rows on this database is useful, but it doesn't prove that two queries
express the same logic for every possible dataset.

You can inspect the [plan and SQL checks](../src/clinical_trial_mvp/agent/analyst.py)
and [reference-result scorer](../src/clinical_trial_mvp/evaluation/scorers.py).


In [6]:
display(JSON(response_example(study, 'baseline')))


<IPython.core.display.JSON object>

### Give the model less schema

Keep the deployment fixed and use the retrieved-schema prompt. Compare first-token
latency and application result matches with the baseline. Inspect what information
was removed before attributing a quality change to the deployment.


In [7]:
result = study.run('schema_retrieved')
print(result.get('_recorded', {}).get('label', study.mode), 'schema_retrieved:', result['status'])
display(Markdown(interpret(study, 'schema_retrieved')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'schema_retrieved')))


recorded Dell measurement schema_retrieved: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/nim/runs/20261107T220801_623823Z.json](../experiment-tracker/mvp4/nim/runs/20261107T220801_623823Z.json)

**Prediction:** Less context may reduce input work but can remove definitions required for correct SQL.

**Observation:** Application result match: 3/4; SQL executed: 4/4. Request E2E p50/p95: 2.735/3.241 s; aggregate output: 40.636 tok/s. Baseline application result match: 2/4.

**Explanation:** Context reduction changes the information available to the model, not just compute cost.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Retain needed clinical definitions; accept reduced context only with adequate task quality.

### Send two requests together

Now increase client concurrency to two. Look at aggregate throughput and the
latency each user experiences. We are changing request load, not the model weights.


In [8]:
result = study.run('concurrency_2')
print(result.get('_recorded', {}).get('label', study.mode), 'concurrency_2:', result['status'])
display(Markdown(interpret(study, 'concurrency_2')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'concurrency_2')))


recorded Dell measurement concurrency_2: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/nim/runs/20261107T220932_908954Z.json](../experiment-tracker/mvp4/nim/runs/20261107T220932_908954Z.json)

**Prediction:** More overlapping clients may improve aggregate throughput while increasing per-request latency.

**Observation:** Application result match: 2/4; SQL executed: 3/4. Request E2E p50/p95: 8.595/12.904 s; aggregate output: 32.502 tok/s.

**Explanation:** Client overlap and scheduler capacity are separate controls; these records do not show achieved batch occupancy.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Choose concurrency using both latency limits and correctly completed work.

### Increase to four concurrent requests

Does the server deliver more useful work with four clients, or mainly longer
queues? Compare with one and two clients, retaining the same deployment and
request settings. Client concurrency is not a direct measurement of GPU batch size.


In [9]:
result = study.run('concurrency_4')
print(result.get('_recorded', {}).get('label', study.mode), 'concurrency_4:', result['status'])
display(Markdown(interpret(study, 'concurrency_4')))
if study.mode == 'plan':
    display(JSON(server_example(study, 'concurrency_4')))


recorded Dell measurement concurrency_4: completed


**recorded Dell measurement** — [experiment-tracker/mvp4/nim/runs/20261107T221207_165140Z.json](../experiment-tracker/mvp4/nim/runs/20261107T221207_165140Z.json)

**Prediction:** More overlapping clients may improve aggregate throughput while increasing per-request latency.

**Observation:** Application result match: 2/4; SQL executed: 3/4. Request E2E p50/p95: 14.085/21.368 s; aggregate output: 39.224 tok/s.

**Explanation:** Client overlap and scheduler capacity are separate controls; these records do not show achieved batch occupancy.

**Limitation:** Four distinct questions, not 20 independent clinical tasks. Repeated requests give illustrative latency summaries; semantic correctness remains unreviewed. Native VRAM is unavailable. These observations do not generalize to other domains or stacks.

**Decision:** Choose concurrency using both latency limits and correctly completed work.

## 5. Separate deployment success from answer quality

The next table brings the observations together. Read transport/request failures,
SQL execution and reference-result matches separately. Request E2E measures the
generation call; application E2E also includes planning, validation, execution and repair.

NIM direct-serving responses were not post-scored as SQL in this recipe, so use
the application-quality columns for SQL outcomes—not transport success. Human
semantic review remains pending. NIM VRAM samples are recorded, but native VRAM
sampling failed; do not use that gap to infer a memory advantage for either path.
This chapter does not supply the controlled cold/warm study from chapter 01.


In [10]:
display(Markdown(study.report()))
display(Markdown(study.cache_report()))


| Configuration/status | SQL | Result match | Semantic review | Prompt tokens | TTFT p50 s | Latency p50/p95 s | tok/s | Peak memory |
|---|---|---|---|---|---|---|---|---|
| MVP4/NIM/qwen/smoke · recorded Dell measurement | 3/4 | 2/4 | Unreviewed; 4 total | 2340 | 2.03 | 6.17/7.57 | 24.23 | 13.95 GiB |
| MVP4/NIM/qwen/baseline · recorded Dell measurement | 3/4 | 2/4 | Unreviewed; 4 total | 2340 | 2.04 | 6.20/7.78 | 24.14 | 13.95 GiB |
| MVP4/NIM/qwen/schema_retrieved · recorded Dell measurement | 4/4 | 3/4 | Unreviewed; 4 total | 458 | 0.19 | 2.73/3.24 | 40.64 | 13.95 GiB |
| MVP4/NIM/qwen/concurrency_2 · recorded Dell measurement | 3/4 | 2/4 | Unreviewed; 4 total | 2340 | 2.09 | 8.59/12.90 | 32.50 | 13.95 GiB |
| MVP4/NIM/qwen/concurrency_4 · recorded Dell measurement | 3/4 | 2/4 | Unreviewed; 4 total | 2340 | 2.14 | 14.09/21.37 | 39.22 | 13.95 GiB |

| Experiment | Application E2E p50/p95 s | Request TTFT p95 s |
|---|---|---|
| smoke · recorded Dell measurement | 12.686/16.845 | 2.034 |
| baseline · recorded Dell measurement | 11.981/16.886 | 2.051 |
| schema_retrieved · recorded Dell measurement | 9.373/9.913 | 0.238 |
| concurrency_2 · recorded Dell measurement | 12.012/16.928 | 3.248 |
| concurrency_4 · recorded Dell measurement | 12.042/16.970 | 6.673 |

No controlled prefix-cache phase measurements in these results.

## 6. Close the study

Save the notebook to retain the displayed outputs. Replay leaves the underlying
measurements unchanged; Live saves new uniquely named records rather than
overwriting the archive.

The live NIM study can reuse its container between successful cells. Run the
next cleanup cell when you finish or pause; don't leave a resident model taking VRAM. The workflow cleans up its own servers on handled errors and interruptions,
but cannot do so after a kernel or OS crash. In that case, use the recorded
process/container identifier to stop only the server this study started.


In [11]:
print(study.close())


Study servers stopped. Saved records and notebook outputs are retained.


### Where to go next

Continue to [Nemotron through NIM](04_nemotron_nim.ipynb), or compare deployment choices in
[the results guide](../tutorial/RESULTS.md#what-should-i-choose).

For setup problems, see the [worked troubleshooting examples](../tutorial/CASE_STUDIES.md).
The [original executed notebook](history/dell-2026-11-07/local%20AI%20optimization%20-%20NIM%20Qwen.ipynb) retains the original run outputs.
The [SQL review packet](../tutorial/SQL_REVIEW.md) and
[GPU workflow checklist](../tutorial/GPU_VALIDATION.md) describe the remaining validation work.
